<a href="https://colab.research.google.com/github/takatakamanbou/MVA/blob/2025/MVA2025_ex10notebookB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVA2025 ex10notebookB

<img width=64 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/MVA/MVA-logo.png"> https://www-tlab.math.ryukoku.ac.jp/wiki/?MVA

----
## 演習課題: データに正規分布を当てはめてみよう + 手計算
---

<font color="#ff0000">
注意:
今回の notebook の中には，コードセルを実行すると問題の解答が表示されるようになっている箇所があります．
</font>


In [ ]:
# 必要なパッケージのインポート
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

# SciPy のもろもろ
from scipy.spatial import distance
from scipy.stats import norm, multivariate_normal

# 解答表示のため
import base64
from IPython.display import display, Markdown

---
### $1.$ データに正規分布を当てはめてみよう

#### $1.1$ データの準備

In [ ]:
##### CSV ファイルを読み込む #####
URL = 'https://www-tlab.math.ryukoku.ac.jp/~takataka/course/MVA/MVA2024-QvsE1107.csv'
df = pd.read_csv(URL, index_col=0)
X = df.to_numpy()
N, D = X.shape
print(f'N = {N}, D = {D}')
df

これは，MVA2024の第7回までの Quiz の得点率 [%]（Quiz）と，小テストの得点率 [%]（Exam）のデータです．小テスト受験者の分のみ含んでいます．

散布図を描くと次のようになります．

In [ ]:
fig, ax = plt.subplots(1, figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], s=10)
ax.set_xlim(-10, 110)
ax.set_ylim(-10, 110)
ax.axhline(0, color='gray')
ax.axvline(0, color='gray')
ax.set_xlabel('Quiz')
ax.set_ylabel('Exam')
ax.set_aspect('equal')
plt.show()

---
#### $1.2$ 平均と分散共分散行列を推定する

「データの準備」で読み込んだ Quiz vs Exam のデータが2次元の正規分布から得られたものであると仮定して，この正規分布パラメータを最尤推定してみましょう．notebookA で説明しているように，最尤推定によるパラメータの解は標本平均と標本分散共分散行列そのものですので，単純にそれらを求めればokです．


データは `X` という名前の NumPy array に格納されています．
また，`N` と `D` がデータの数（サンプルサイズ）と次元数を表します．

In [ ]:
print('(X の最初の5行) = ')
print(X[:5, :])
print(X.shape)
print(f'N = {N}, D = {D}')

#### 問題1

(1) 次のコードセルに，`X` の行方向の平均（Quizの平均とExamの平均）を求めて `mu` という変数に代入し，それを print するコードを書きなさい．NumPy の関数 np.mean を用いること．オプション引数 `axis` の指定が必要ですね．

In [ ]:
# このセルを実行すると上記の解答例を表示します
Q = b'CmBgYAptdSA9IG5wLm1lYW4oWCwgYXhpcz0wKQpwcmludChtdSkKYGBgCg=='
display(Markdown(base64.b64decode(Q).decode('utf-8')))

(2) 次のコードセルに，`X` の分散共分散行列を求めて `cov` という変数に代入し，それを print するコードを書きなさい．NumPy の関数を使うこともできますが，ここでは，次のヒントを参考にして自分で式を書いてみましょう．

［ヒント］ ベクトルを **列ベクトル** として扱うものとして，$N$ 個の $D$ 次元列ベクトルを $\pmb{x}_1, \pmb{x}_2, \ldots, \pmb{x}_N$ とおき，$\pmb{\mu} = \frac{1}{N}\sum_{n=1}^N \pmb{x}_n$ とする．これらの分散共分散行列を $\Sigma$ とおくと，

$$
\Sigma = \frac{1}{N}\sum_{n=1}^N (\pmb{x}_n - \pmb{\mu}) (\pmb{x}_n - \pmb{\mu})^{\top}
$$

である．このとき，$D \times N$ 行列 $X'$ を $X' = \begin{pmatrix} \pmb{x}_1 - \pmb{\mu} & \pmb{x}_2 - \pmb{\mu} & \cdots & \pmb{x}_N - \pmb{\mu} \end{pmatrix}$ とおくと，

$$
\Sigma = \frac{1}{N} X' X'^{\top}
$$

となる．この notebook では，ベクトルを **行ベクトル** として扱い，変数 `X` は $\pmb{x}_1, \ldots, \pmb{x}_N$ をならべた $N \times D$ 行列を表している．この場合，`X - mu` が上記の行列 $X'$ に対応する．

In [ ]:
# このセルを実行すると上記の解答例を表示します
Q = b'CmBgYApYZCA9IFggLSBtdQpjb3YgPSBYZC5UIEAgWGQgLyBOCnByaW50KGNvdikKYGBgCg=='
display(Markdown(base64.b64decode(Q).decode('utf-8')))

`X` に格納されている値は，0 列目が Quiz，1列目が Exam です．したがって，`cov` を正しく求めた場合，`cov[0, 0]` が Quiz の分散，`cov[1, 1]` が Exam の分散，`cov[0, 1]` と `cov[1, 0]` が Quiz と Exam の共分散を表します．したがって，両者の相関係数は次のように計算できます．


In [ ]:
rho = cov[0, 1] / np.sqrt(cov[0, 0]*cov[1, 1])
print(rho)

#### $1.3$ 散布図に確率密度関数を重ねて描画する + マハラノビス距離を求める

上で求めた平均と分散共分散行列は，Quiz vs Exam のデータが2次元正規分布から得られていると仮定したときに，そのパラメータ（平均と分散共分散行列）を最尤推定によって求めたものとなっています．
得られた正規分布の確率密度関数を $f(\pmb{x})$ とおくとき，いくつかの定数 $c$ に対して $f(\pmb{x}) = c$ となる点（マハラノビス距離が一定となる点）の座標を求め，データの散布図に重ねて描いてみると，次のようになります．



In [ ]:
# 距離計算の対象にする点の座標
P = np.array([40, 40])

# 確率密度描画のためのグリッドデータの作成
xmin, xmax = -10, 110
xx, yy = np.mgrid[xmin:xmax:1.0, xmin:xmax:1.0]
zz = multivariate_normal.pdf(np.dstack((xx, yy)), mean=mu, cov=cov)

# データの散布図と正規分布の確率密度関数を重ねて描く
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], s=10)
ax.contour(xx, yy, zz, colors=['#ffa0a0', '#ff5050', '#ff0000'], levels=[0.00002, 0.0001, 0.0005])
ax.plot(mu[0], mu[1], '+', markersize=16, color='r')
ax.scatter(P[0], P[1], s=150, marker='*', label=f'({P[0]},{P[1]})')
ax.axvline(mu[0], linestyle='dotted', color='gray')
ax.axhline(mu[1], linestyle='dotted', color='gray')
ax.axvline(0, color='gray')
ax.axhline(0, color='gray')
ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
ax.set_xlabel('Quiz')
ax.set_ylabel('Exam')
ax.set_aspect('equal')
ax.legend()
plt.show()

# 点 P と推定された正規分布の平均との間のユークリッド距離，および点 P のこの正規分布に対するマハラノビス距離
covInv = np.linalg.inv(cov)
tmp = P - mu
edist = np.sqrt(tmp @ tmp)
mdist = np.sqrt(tmp @ covInv @ tmp)
print(f'平均との間のユークリッド距離 = {edist:.2f}   正規分布との間のマハラノビス距離 = {mdist:.2f}')

#### 問題2

上のコードセルを実行すると，$(\text{Quiz}, \text{Exam}) = (40, 40)$ の点と推定された正規分布の平均との間のユークリッド距離，および，この点と推定された正規分布との間のマハラノビス距離の値が表示されます．コードセルの2行目を書き換えて $(95, 50)$ の場合のこれらの距離の値を求め，$(40, 40)$ と $(95, 50)$ ではどちらの方がこの正規分布から生成されたデータとして尤もらしいか答えなさい．

---
### $2.$ マハラノビス距離を手計算してみよう

#### 問題3

3つの2次元ベクトル（$2\times 1$ 行列） $\pmb{x}, \pmb{y}, \pmb{\mu}$ を

$$
\pmb{x} = \begin{pmatrix} 4 \\ 3 \end{pmatrix}\qquad
\pmb{y} = \begin{pmatrix} 5 \\ 5 \end{pmatrix}\qquad
\pmb{\mu} = \begin{pmatrix} 3 \\ 4 \end{pmatrix}
$$

とおく．また，行列 $\Sigma$ を

$$
\Sigma = \begin{pmatrix} 2 & 1 \\ 1 & 1 \end{pmatrix}
$$

とおく．このとき，次の問に答えなさい．

(1) $\pmb{x}$ と $\pmb{\mu}$ との間のユークリッド距離 $\| \pmb{x} - \pmb{\mu}\|$ および $\pmb{y}$ と $\pmb{\mu}$ との間のユークリッド距離 $\|\pmb{y}-\pmb{\mu}\|$ を求め，この距離規準では $\pmb{x}$ と $\pmb{y}$ のどちらの方が $\pmb{\mu}$ に近いか答えなさい．

(2) 平均 $\pmb{\mu}$，分散共分散行列 $\Sigma$ の正規分布に対する $\pmb{x}, \pmb{y}$ のマハラノビス距離 $d(\pmb{x}), d(\pmb{y})$ を求め，この距離規準では $\pmb{x}$ と $\pmb{y}$ のどちらの方がこの正規分布に近いか（この正規分布から生成されたデータとして尤もらしいか）答えなさい．



次のコードを実行すると，上記の略解を表示します．また，上記問題の状況を視覚的に表示させることができます．

In [ ]:
# 正規分布の平均と分散共分散行列
mu2 = np.array([3.0, 4.0])
cov2 = np.array([[2.0, 1.0], [1.0, 1.0]])
# 2点 R, S
R = np.array([4, 3])
S = np.array([5, 5])

fig, ax = plt.subplots(figsize=(6, 6))
xmin, xmax = -1, 7
ymin, ymax = -1, 7
xx, yy = np.mgrid[xmin:xmax:0.1, xmin:xmax:0.1]
# 等高線を描くマハラノビス距離の値
levels = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
X2 = np.dstack((xx, yy)).reshape((-1, 2))
cov2Inv = np.linalg.inv(cov2)
d2 = np.sum(((X2 - mu2) @ cov2Inv * (X2 - mu2)), axis=1)
zz = np.sqrt(d2).reshape((xx.shape[0], yy.shape[0]))
cs = ax.contour(xx, yy, zz, cmap='Blues_r', levels=levels)
ax.clabel(cs)
ax.scatter(R[0], R[1], s=100, marker='*', label=r'$\mathbf{x}$')
ax.scatter(S[0], S[1], s=100, marker='*', label=r'$\mathbf{y}$')
ax.scatter(mu2[0], mu2[1], s=100, marker='+', color='blue', label=r'$\mathbf{\mu}$')
ax.axhline(0, color='gray')
ax.axvline(0, color='gray')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')
ax.legend()
plt.show()

# ユークリッド距離とマハラノビス距離
tmp = R - mu2
edistR = np.sqrt(tmp @ tmp)
mdistR = np.sqrt(tmp @ cov2Inv @ tmp)
tmp = S - mu2
edistS = np.sqrt(tmp @ tmp)
mdistS = np.sqrt(tmp @ cov2Inv @ tmp)
print(f'点 x と平均とのユークリッド距離 = {edistR:.4f}   点 x とこの正規分布とのマハラノビス距離 = {mdistR:.4f}')
print(f'点 y と平均とのユークリッド距離 = {edistS:.4f}   点 y とこの正規分布とのマハラノビス距離 = {mdistS:.4f}')

Q = b'CigxKSAkXHwgXHBtYnt4fSAtIFxwbWJ7XG11fVx8ID0gXHNxcnR7KDQtMyleMisoMy00KV4yfSA9IFxzcXJ0ezJ9LFwgIFx8IFxwbWJ7eX0gLSBccG1ie1xtdX1cfCA9IFxzcXJ0ezV9JO+8jiRccG1ie3h9JCDjga7mlrnjgYzov5HjgYTvvI4KCigyKSAkXFNpZ21hXnstMX0gPSBcYmVnaW57cG1hdHJpeH0gMSAmIC0xIFxcIC0xICYyIFxlbmR7cG1hdHJpeH0kIOOCiOOCiu+8jAokZF4yKFxwbWJ7eH0pID0gXGJlZ2lue3BtYXRyaXh9IDQgLSAzICYgMyAtIDRcZW5ke3BtYXRyaXh9XGJlZ2lue3BtYXRyaXh9IDEgJiAtMSBcXCAtMSAmMiBcZW5ke3BtYXRyaXh9XGJlZ2lue3BtYXRyaXh9IDQtMyBcXCAzLTQgXGVuZHtwbWF0cml4fSA9IDUk77yM44KI44Gj44GmICRkKFxwbWJ7eH0pID0gXHNxcnR7NX0k77yOCuWQjOanmOOBq++8jCRkKFxwbWJ7eX0pID0gXHNxcnR7Mn0k77yOJFxwbWJ7eX0kIOOBruaWueOBjOi/keOBhO+8jgo='
display(Markdown(base64.b64decode(Q).decode('utf-8')))

---
### $3.$ 1次元正規分布のパラメータの最尤推定の解を導出してみよう

#### 問題4

(1) notebookA の式$(*)$ を $\mu$ で微分したもの，つまり $\frac{\partial L}{\partial \mu}$ を求めなさい．

(2) $\frac{\partial L}{\partial \mu} = 0$ を $\mu$ について解いて，notebookA の式$(1)$が得られることを示しなさい．



In [ ]:
# このセルを実行すると上記の略解を表示します
Q = b'CigxKQokJApcYmVnaW57YWxpZ25lZH0KXGZyYWN7XHBhcnRpYWwgTH17XHBhcnRpYWwgXG11fSAmPSAtXGZyYWN7Mn17MlxzaWdtYV4yfVxzdW1fe249MX1ee059KHhfbiAtIFxtdSkoLTEpID0gXGZyYWN7MX17XHNpZ21hXjJ9XHN1bV97bj0xfV57Tn0oeF9uIC0gXG11KVxcCiY9IFxmcmFjezF9e1xzaWdtYV4yfVxsZWZ0KFxzdW1fe249MX1ee059eF9uIC0gTlxtdVxyaWdodCkKXGVuZHthbGlnbmVkfQokJAoKKDIpIOecgeeVpQo='
display(Markdown(base64.b64decode(Q).decode('utf-8')))

(3) notebookA の式$(*)$ を $\sigma^2$ で微分したもの，つまり $\frac{\partial L}{\partial \sigma^2}$ を求めなさい．ここでは，$\sigma$ではなく$\sigma^2$ をひとかたまりの変数として扱っていることに注意．

(4) $\frac{\partial L}{\partial \sigma^2} = 0$ を $\sigma^2$ について解いて，notebookA の式$(2)$が得られることを示しなさい．

In [ ]:
# このセルを実行すると上記の略解を表示します
Q = b'CigzKQokJApcYmVnaW57YWxpZ25lZH0KXGZyYWN7XHBhcnRpYWwgTH17XHBhcnRpYWwgXHNpZ21hXjJ9ICY9IC1cZnJhY3tOfXsyfVxmcmFjezF9e1xzaWdtYV4yfSArIFxmcmFjezF9ezIoXHNpZ21hXjIpXjJ9XHN1bV97bj0xfV57Tn0oeF9uIC0gXG11KV4yIFxcCiY9IFxmcmFjezF9ezIoXHNpZ21hXjIpXjJ9XGxlZnQoXHN1bV97bj0xfV57Tn0oeF9uIC0gXG11KV4yIC0gTlxzaWdtYV4yXHJpZ2h0KSBcXApcZW5ke2FsaWduZWR9CiQkCgooNCkg55yB55WlCg=='
display(Markdown(base64.b64decode(Q).decode('utf-8')))